# LGBM + GraphSAGE two-stage cascade
Uses frozen seed02 models. Selects thresholds from full validation, then evaluates full test.


In [ ]:
from pathlib import Path
import json, os, sys, tarfile

MODEL_NAME = 'lgbm'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUTPUT = WORK / f'cascade_run_v3_{MODEL_NAME}'
WORK.mkdir(parents=True, exist_ok=True)

# Kaggle may expose the dataset as extracted files or as a single archive.
bundle = INPUT / 'datasets/nhatnguyen2101/optc-cascade-bundle'
required = ('bundle_manifest.json', 'provenance/graph.sqlite', 'provenance/model_numpy.npz',
            'code/cascade_runtime.py', f'models/{MODEL_NAME}_metrics.json')
if not all((bundle / name).is_file() for name in required):
    extracted = [p.parent for p in INPUT.glob('**/bundle_manifest.json')
                 if all((p.parent / name).is_file() for name in required)]
    if len(extracted) == 1:
        bundle = extracted[0]
    else:
        archives = sorted(INPUT.glob('**/provenance-cascade-bundle-v3.tar.gz'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted v3 bundle or archive; found {len(extracted)} extracted, {len(archives)} archives')
        bundle = WORK / 'cascade_bundle_v3'
        bundle.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r:gz') as tar:
            for member in tar.getmembers():
                target = (bundle / member.name).resolve()
                if not target.is_relative_to(bundle.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe bundle member: {member.name}')
            tar.extractall(bundle)
if json.loads((bundle / 'bundle_manifest.json').read_text()).get('schema') != 3:
    raise RuntimeError('RF, XGB and LGBM notebooks require the v3 cascade bundle')
sys.path.insert(0, str(bundle / 'code'))
from cascade_runtime import run

host_root = INPUT / 'datasets/nhatnguyen2101/optc-host-ml-full-input'
manifest = host_root / 'manifest.json'
if not (manifest.is_file() and (host_root / 'assets/vocab.json').is_file()):
    candidates = [p for p in INPUT.glob('**/manifest.json')
                  if (p.parent / 'assets/vocab.json').is_file()]
    if len(candidates) == 1:
        manifest = candidates[0]
    else:
        archives = sorted(INPUT.glob('**/host-input.tar'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted Host dataset or archive; found {len(candidates)} extracted, {len(archives)} archives')
        host_root = WORK / 'host_shards'
        host_root.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r') as tar:
            for member in tar.getmembers():
                target = (host_root / member.name).resolve()
                if not target.is_relative_to(host_root.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe Host archive member: {member.name}')
            tar.extractall(host_root)
        manifest = host_root / 'manifest.json'

print('Host model:', MODEL_NAME)
print('Cascade bundle:', bundle)
print('Host shards:', manifest)
print('Output:', OUTPUT)
result = run(manifest, bundle, OUTPUT, model_name=MODEL_NAME)
print('Results ZIP:', OUTPUT / 'optc-two-stage-cascade-results.zip')


In [ ]:
# Measured validation/test diagnostics, after the full cascade finishes.
import matplotlib.pyplot as plt
import numpy as np
import zipfile
from IPython.display import display, Image

cal = json.loads((OUTPUT / 'validation_calibration.json').read_text())
measured = json.loads((OUTPUT / 'two_stage_cascade_metrics.json').read_text())
base, fused = measured['baseline'], measured['cascade']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, title, report in zip(axes, ('Frozen Host ML', 'Two-stage cascade'), (base, fused)):
    c = report['confusion']
    matrix = np.array([[c['tn'], c['fp']], [c['fn'], c['tp']]])
    ax.imshow(np.log1p(matrix), cmap='Blues')
    for row in range(2):
        for col in range(2):
            ax.text(col, row, f'{matrix[row, col]:,}', ha='center', va='center',
                    bbox={'boxstyle':'round', 'facecolor':'white', 'alpha':0.85})
    ax.set(xlabel='Predicted', ylabel='Actual', title=title,
           xticks=[0, 1], yticks=[0, 1], xticklabels=['Benign', 'Malicious'],
           yticklabels=['Benign', 'Malicious'])
fig.suptitle(f'{MODEL_NAME.upper()} + GraphSAGE: full test confusion (log color scale)')
fig.tight_layout()
confusion_path = OUTPUT / 'confusion_matrix_comparison.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

fig, (ax, route_ax) = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ['Precision', 'Recall', 'F1']
base_values = [base['precision'], base['recall'], base['f1']]
fused_values = [fused['precision'], fused['recall'], fused['f1']]
positions = np.arange(3)
ax.bar(positions - 0.18, base_values, width=0.36, label='Host ML')
ax.bar(positions + 0.18, fused_values, width=0.36, label='Two-stage')
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 1.05), ylabel='Test score', title='Full test metrics')
ax.legend()
route_ax.bar(['Direct', 'Graph'], [fused['rows'] - measured['routed'], measured['routed']])
route_ax.set(ylabel='Test events', title='Routing by validation-selected policy')
fig.tight_layout()
comparison_path = OUTPUT / 'metric_and_routing_comparison.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

print('Validation policy:', cal['selected'])
print(f"Validation F1: {next(x['f1'] for x in cal['grid'] if x['policy'] == cal['selected']):.4%}")
print(f"Test F1: {fused['f1']:.4%}; baseline: {base['f1']:.4%}")
print(f"Routed to GraphSAGE: {measured['routed']:,} / {fused['rows']:,}")
print(f"Missing graph lookups: {measured['missing_lookup']:,}")
source_index_path = OUTPUT / 'source_index.json'
source_index_path.write_text(json.dumps({s['source_id']: s['source']['path'] for s in json.loads(manifest.read_text())['sources']}, indent=2))
with zipfile.ZipFile(OUTPUT / 'optc-two-stage-cascade-results.zip', 'a', zipfile.ZIP_DEFLATED) as archive:
    archive.write(confusion_path, arcname=confusion_path.name)
    archive.write(comparison_path, arcname=comparison_path.name)
    archive.write(source_index_path, arcname=source_index_path.name)
print('Saved plots to', OUTPUT)
